# Wilson-Cowan: excitatory and inhibitory populations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZetaOne/pyauto-xpp/blob/main/tutorials/notebooks/08_wilson_cowan.ipynb)

Mean activities `u` (excitatory) and `v` (inhibitory) of two coupled neural populations
with a sigmoid response. Depending on the thresholds and weights the network rests,
switches between states, or oscillates. The branches of equilibria have sharp folds, so
AUTO needs a small step.

*Based on an example from XPP Studio (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.2 and its AUTO through pyauto-xpp.*

Running in Google Colab? This cell installs pyauto-xpp first; elsewhere it does nothing.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip -q install pyauto-xpp "plotly>=6" anywidget ipywidgets --find-links https://github.com/ZetaOne/pyauto-xpp/releases/expanded_assets/v0.3.0
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ZetaOne/pyauto-xpp/main/tutorials/notebooks/xpptut.py", "xpptut.py")
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [3]:
mb = (ModelBuilder("wilson_cowan")
      .comment("Wilson-Cowan model")
      .function("f(u)", "1/(1+exp(-u))")
      .ode("u", "(-u+f(aee*u-aei*v-te))/taue")
      .ode("v", "(-v+f(aie*u-aii*v-ti))/taui")
      .parameter(aee=13, aei=10, te=2, taue=2)
      .parameter(aie=16, aii=4, ti=5, taui=6)
      .initial(u=0.1, v=0.1)
      .option(total=200, ds=0.01))
print(mb.render())

# Wilson-Cowan model
f(u)=1/(1+exp(-u))
u'=(-u+f(aee*u-aei*v-te))/taue
v'=(-v+f(aie*u-aii*v-ti))/taui
par aee=13,aei=10,te=2,taue=2
par aie=16,aii=4,ti=5,taui=6
init u=0.1,v=0.1
@ total=200,ds=0.01
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [4]:
ex = Explorer(mb, kind="plane", x="u", y="v", window=(-0.1, 1.1, -0.1, 1),
              sliders={"te": (-2, 6), "ti": (0, 8), "aee": (5, 20), "taui": (1, 15)},
              auto="te", auto_range=(-4, 6), auto2="ti", auto2_range=(0, 10), auto_ds=0.002, auto_dsmax=0.01,
              title="Wilson-Cowan")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Equilibria along the excitatory threshold `te`.** The S-shaped branch has folds;
between them several states coexist.

In [5]:
parts = ex.run_auto()
for d in parts:
    print([(p.type, round(p.x, 4)) for p in d.labeled() if p.type in ("HB", "LP")])

[('HB', 3.1711)]
[('LP', -0.9926), ('LP', -0.4457)]


## Things to try

- Slide `taui` up: inhibition gets slower and the network starts to oscillate.
- Try the **XPP direct field** and **arrows coloured by speed**: the dynamics are very slow near the folds.